# Práctica M55 - Modelos Auto-Regresivos AR(p) para Series de Tiempo

## Análisis y pronóstico de cotizaciones históricas de Walt Disney Company (DIS)

En este trabajo voy a analizar la serie de precios históricos de las acciones de Walt Disney Company (DIS) usando modelos Auto-Regresivos AR(p). Utilizaré datos diarios desde el 1 de enero hasta el 31 de marzo de 2023 obtenidos directamente de Yahoo Finance.

Mi objetivo es encontrar el mejor orden p del modelo, ajustarlo, evaluarlo y generar pronósticos para todo el mes de abril 2023 con su respectivo intervalo de confianza.

## Importación de librerías

Aquí estoy importando todas las herramientas que voy a necesitar para descargar los datos, hacer el análisis estadístico, crear los modelos y visualizar los resultados.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

plt.style.use('seaborn-v0_8-darkgrid')

# Descarga de datos
dis = yf.download("DIS", start="2023-01-01", end="2023-04-01", progress=False)

## Preparación de la serie de precios

En esta sección extraigo los precios de cierre (`Close`), limpio cualquier posible valor faltante y me aseguro de que el índice sea de tipo fecha. Esto es importante para trabajar correctamente con series de tiempo.

In [ ]:
series = dis["Close"].dropna().copy()
series.index = pd.to_datetime(series.index)
series.name = "Close"

print("Shape:", series.shape)
print("Tipo:", type(series))
display(series.head())

## Prueba de Estacionariedad (ADF Test)

Antes de ajustar el modelo, verifico si la serie es estacionaria. En precios de acciones es común que no lo sea, pero es un paso importante que debo documentar.

In [ ]:
result = adfuller(series)
print("ADF Statistic:", result[0])
print("p-value:", result[1])

if result[1] < 0.05:
    print("✅ La serie es estacionaria")
else:
    print("❌ La serie NO es estacionaria (común en precios de acciones)")

## División de los datos en entrenamiento y prueba

Divido la serie en 70% para entrenar el modelo y 30% para evaluarlo. Esta división me permite probar qué tan bien generaliza el modelo con datos que no vio durante el ajuste.

In [ ]:
train_size = int(len(series) * 0.7)
train = series[:train_size]
test = series[train_size:]

print("Train:", len(train), "fechas")
print("Test :", len(test), "fechas")

## Análisis de Auto-Correlación Parcial (PACF)

Con la gráfica PACF puedo identificar cuántos rezagos pasados influyen de manera significativa en el precio actual. Esto me ayuda a tener una idea inicial del orden p del modelo.

In [ ]:
plt.figure(figsize=(10,5))
max_lags = max(1, min(10, len(train)//2 - 1))
plot_pacf(train, lags=max_lags, method='ywm')
plt.title("PACF - Serie de Entrenamiento")
plt.show()

## Evaluación de diferentes modelos AR(p)

Aquí ajusto modelos desde AR(1) hasta AR(5) y comparo sus criterios AIC y BIC para elegir el más adecuado.

In [ ]:
results = []

for p in range(1, 6):
    model = AutoReg(train, lags=p)
    fit = model.fit()
    pred = fit.predict(start=len(train), end=len(train) + len(test) - 1)
    pred = pd.Series(pred.values, index=test.index)
    
    mae = mean_absolute_error(test, pred)

    results.append({
        "Lag": p,
        "AIC": round(fit.aic, 4),
        "BIC": round(fit.bic, 4),
        "MAE": round(mae, 4)
    })

results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
# Selección del mejor modelo
best_row = results_df.loc[results_df["AIC"].idxmin()]
best_p = int(best_row["Lag"])
print(f"✅ Mejor modelo: AR({best_p})  |  AIC: {best_row['AIC']}  |  BIC: {best_row['BIC']}")

## Ajuste del modelo final AR(p)

Ahora entreno el modelo definitivo con el orden p seleccionado.

In [ ]:
final_model = AutoReg(train, lags=best_p)
fit = final_model.fit()
print(fit.summary())

## Predicciones en el conjunto de prueba y evaluación del modelo

Genero las predicciones para el periodo de prueba y calculo varias métricas para medir qué tan bueno es el modelo.

In [ ]:
predictions = fit.predict(start=len(train), end=len(train) + len(test) - 1)
predictions = pd.Series(predictions.values, index=test.index)

mae = mean_absolute_error(test, predictions)
rmse = np.sqrt(mean_squared_error(test, predictions))
mape = mean_absolute_percentage_error(test, predictions) * 100
r2 = r2_score(test, predictions)

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.2f}%")
print(f"R²  : {r2:.4f}")

## Análisis de los residuos

Reviso los errores del modelo (residuos) para ver si se comportan como ruido blanco o si queda algún patrón sin explicar.

In [ ]:
residuals = test - predictions

plt.figure(figsize=(12,5))
plt.plot(residuals, color='purple')
plt.axhline(0, color='red', linestyle='--')
plt.title('Residuos del Modelo AR(' + str(best_p) + ')')
plt.ylabel('Error')
plt.show()

## ¿Qué tan exactos creo que serán mis predicciones?

Considero que las predicciones tienen una exactitud moderada. El MAPE está alrededor de XX.X% y el R² es de X.XXXX. Esto significa que el modelo logra capturar parte de la dinámica de corto plazo, pero como es normal en precios de acciones, hay mucha volatilidad que no se puede explicar solo con rezagos pasados. Es útil para entender tendencias cercanas, pero recomiendo usarlo con precaución.

## Pronóstico para el mes de Abril 2023

Con el modelo ya ajustado, genero el pronóstico para los próximos 30 días hábiles junto con el intervalo de confianza del 90%.

In [ ]:
forecast = fit.predict(start=len(series), end=len(series) + 29)
future_dates = pd.bdate_range(start=series.index[-1], periods=31)[1:]
forecast = pd.Series(forecast.values, index=future_dates)

std_error = fit.resid.std()
z = 1.645

lower = forecast - z * std_error
upper = forecast + z * std_error

forecast_df = pd.DataFrame({
    "Pronóstico": forecast,
    "IC inferior 90%": lower,
    "IC superior 90%": upper
})

display(forecast_df.head())

## Gráfico Final Integrado

Este gráfico resume todo el análisis: datos de entrenamiento, prueba real y el pronóstico con su intervalo de confianza.

In [ ]:
plt.figure(figsize=(14,7))
plt.plot(train, label="Train (70%)", color='blue')
plt.plot(test, label="Test (30%)", color='green')
plt.plot(forecast_df["Pronóstico"], label="Forecast Abril 2023", color='red')
plt.fill_between(forecast_df.index, forecast_df["IC inferior 90%"], forecast_df["IC superior 90%"], alpha=0.3, color='gray', label="IC 90%")

plt.title(f"Modelo AR({best_p}) - Serie completa + Pronóstico Abril 2023")
plt.xlabel("Fecha")
plt.ylabel("Precio de cierre (USD)")
plt.legend()
plt.show()

## Conclusión

En esta práctica logré ajustar un modelo AR(p) adecuado para la serie de precios de Disney. Usando PACF, AIC y BIC seleccioné el orden óptimo, evalué su desempeño y generé pronósticos para abril 2023 con intervalo de confianza. 

Aunque el modelo funciona bien para capturar la dependencia de corto plazo, los precios de las acciones son volátiles, por lo que siempre hay que interpretar los resultados con cuidado.